In [1]:
# pip install ipykernel ipywidgets
# pip install datasets langchain langchain-openai python-dotenv

# 1. HuggingFace Datasets 패키지

- HuggingFace Datasets는 머신러닝 모델 학습에 필요한 대규모 데이터셋을 효율적으로 로드, 전처리, 변환, 스트리밍할 수 있도록 지원하는 라이브러리이다.
- HuggingFace Hub에 등록된 다양한 데이터셋을 동일한 인터페이스로 접근할 수 있다.
- 설치

  - `pip install datasets`
- 공식 문서

  - [Dataset Hub: https://huggingface.co/datasets](https://huggingface.co/datasets)
  - [https://github.com/huggingface/datasets](https://github.com/huggingface/datasets)
  - [https://huggingface.co/docs/datasets/index](https://huggingface.co/docs/datasets/index)
## 1.1 Dataset과 DatasetDict

### 1.1.1 Dataset

- 하나의 데이터셋을 테이블 형태(행 = 샘플, 열 = Feature)로 저장하는 구조.
- 텍스트, 이미지, 오디오, 멀티모달 데이터 지원.
- PyTorch, TensorFlow 학습 파이프라인과 연동 가능.
- Lazy Loading 및 Streaming을 통해 대규모 데이터도 메모리 효율적으로 처리.

### 1.1.2 DatasetDict

- 여러 Dataset을 key-value 형태로 묶어 관리한다.

  - 예: `{"train": Dataset, "validation": Dataset, "test": Dataset}`
- key를 통해 데이터셋에 접근.

## 1.2 DatasetDict / Dataset 로딩
- `datasets.load_dataset()` 이용
  
### 1.2.1 Hub 데이터 로딩
- Huggingface hub에 등록된 Dataset의 id를 전달하면  다운로드 후 로딩한다.

```python
from datasets import load_dataset
dataset_dict = load_dataset("imdb")
```
![img](figures/huggingface_dataset.png)

### 1.2.2 로컬 파일 로딩

```python
dataset_dict = load_dataset(
    "csv",
    data_files={
        "train": "data/train.csv",
        "validation": "data/valid.csv",
        "test": "data/test.csv"
    }
)
```

### 1.2.3 다양한 소스에서 Dataset 생성

```python
from datasets import Dataset

Dataset.from_pandas(df) # DataFrame을 Dataset으로 로딩
Dataset.from_dict({"text": [...], "label": [...]}) # 딕셔너리를 Dataset으로 로딩
Dataset.from_csv("path.csv") # 경로의 csv파일을 Dataset으로 로딩
```

## 1.3 Dataset 주요 기능

### 1.3.1 데이터 접근(조회)

- 인덱싱으로 샘플 조회: `dataset[0]`
- 슬라이싱: `dataset[:10]`

### 1.3.1 전처리 관련 메소드

#### 1.3.1.1 map()

데이터셋의 모든 데이터(또는 지정한 batch)에 변환 함수를 적용한다.

```python
dataset.map(
    function,                    # 각 샘플/배치에 적용할 함수
    batched=False,               # True: 배치 단위 입력
    batch_size=1000,             # batched=True일 때 배치사이즈. 기본값: 1000
    remove_columns=None,         # 변환 후 제거할 컬럼 리스트
    num_proc=None,               # 병렬 처리 프로세스 개수
)
```

##### 1.3.1.1.1 map에 전달하는 함수(function)의 입력(parameter)/출력(return) 형태
- **parameter: dict** - Dataset의 데이터를 dict(컬럼명, 값) 로 받는다. batched=False일 경우 개별 값, True일 경우 값은 iterable
- **return: dict** - 처리한 결과. dict(컬럼명, 처리값) 지정한 컬럼에 처리한 값을 저장한다. 컬럼명이 **기존 컬럼이면 덮어쓰고, 없는 컬럼이면 추가**한다. `remove_columns=컬럼명` 옵션을 지정하면 처리가 끝난 뒤 지정한 컬럼을 제거할  수 있다.
- `batched=False` (기본값) 일 때

  - 함수 입력: **dict** (**하나의 row**)
  - 함수 반환: **dict** (**추가될 컬럼 포함**)

    ```python
    def add_length(example):
        example["text_len"] = len(example["text"])
        return example

    dataset = dataset.map(add_length)
    ```

- `batched=True` 일때 (배치단위 입력)

  - 함수 입력: **dict** (**여러 row의 컬럼이 리스트로 묶여 들어옴**)
  - 함수 반환: **dict** (**리스트 형태 유지**)

    ```python
    def add_length_batch(batch):
        return {"text_len": [len(t) for t in batch["text"]]}

    dataset = dataset.map(add_length_batch, batched=True)
    ```

#### 1.3.1.2 filter()

조건 함수를 적용하여 True를 반환하는 row만 남긴다.

```python
dataset.filter(
    function,           # 조건 함수
    batched=False,      # 배치 처리 여부
    num_proc=None,      # 병렬 처리
    desc=None
)
```

##### 1.3.1.2.1 filter()에 전달하는 함수의 입력(parameter)/출력(return) 형태
- **parameter: dict** - Dataset의 데이터를 dict(컬럼명, 값) 로 받는다. batched=False일 경우 개별 값, True일 경우 값은 iterable
- **return: bool** -  체크한 Datapoint를 Dataset에 남길지 말지 여부
- `batched=False`: 한 줄씩 조건 적용

  ```python
  dataset = dataset.filter(lambda x: x["label"] == 1)
  ```

- `batched=True`: 배치 활용

  ```python
  def filter_positive(batch):
      return [label == 1 for label in batch["label"]]

  dataset = dataset.filter(filter_positive, batched=True)
  ```

## 1.4 기타 유용 메소드

- **컬럼 삭제**
  - `dataset = dataset.remove_columns(["column_name"])`
- **컬럼 이름 변경**
  - `dataset = dataset.rename_column("orig", "new")`
- **데이터 분할**
  - `dataset.train_test_split(test_size=0.2)`

- **저장**

  ```python
  dataset.save_to_disk("path/")
  dataset = datasets.load_from_disk("path/")
  ```

- 모델학습 프레임워크용 포맷 변환
  - 기본적으로 Dataset의 데이터들은 list 로 저장됨.
  - pytorch나 tensorflow에서 사용할 수있도록 format을 변환. (값을 제공할 때 지정한 타입으로 변환해서 반환한다.)

    ```python
    dataset.set_format(type="torch") # numpy, torch, tensorflow, python
    ```

- Streaming 로딩
    - Huggingface Hub의 데이터셋이 너무 클 경우(수백 GB) 필요한 데이터만 순차적으로 가져와 로딩하여 처리할 수있다.
      ```python
      ds_stream = load_dataset("c4", split="train", streaming=True)

      for sample in ds_stream:
        sample 로 사용
      ```


# 2. Dataset 예제
- https://huggingface.co/datasets/LLM-SocialMedia/Korean-YouTube-Comment-Sentiment-Dataset
- Dataset ID: LLM-SocialMedia/Korean-YouTube-Comment-Sentiment-Dataset

In [2]:
####################################################
# DataLoading - load_dataset("Datahub Dataset ID")
####################################################

from datasets import load_dataset

dataset_dict = load_dataset("LLM-SocialMedia/Korean-YouTube-Comment-Sentiment-Dataset")

print(type(dataset_dict))

<class 'datasets.dataset_dict.DatasetDict'>


In [3]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['cid', 'text', 'time', 'author', 'channel', 'votes', 'replies', 'heart', 'reply', 'time_parsed', 'source_video_id', 'source_video_title', 'text_length', 'is_not_spam', 'votes_norm', 'replies_norm', 'heart_norm', '검수_감정'],
        num_rows: 10035
    })
})

In [4]:
#####################################
# DatasetDict에서 Dataset 조회
#####################################
dataset = dataset_dict['train']
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [5]:
dataset

Dataset({
    features: ['cid', 'text', 'time', 'author', 'channel', 'votes', 'replies', 'heart', 'reply', 'time_parsed', 'source_video_id', 'source_video_title', 'text_length', 'is_not_spam', 'votes_norm', 'replies_norm', 'heart_norm', '검수_감정'],
    num_rows: 10035
})

In [6]:
#################################
# Dataset의 정보 조회
#################################
dataset.features
# key: feature 이름, value: data type
# test : 댓글, 검수_감정: label

{'cid': Value('string'),
 'text': Value('string'),
 'time': Value('string'),
 'author': Value('string'),
 'channel': Value('string'),
 'votes': Value('int64'),
 'replies': Value('int64'),
 'heart': Value('bool'),
 'reply': Value('bool'),
 'time_parsed': Value('float64'),
 'source_video_id': Value('string'),
 'source_video_title': Value('string'),
 'text_length': Value('int64'),
 'is_not_spam': Value('bool'),
 'votes_norm': Value('float64'),
 'replies_norm': Value('float64'),
 'heart_norm': Value('int64'),
 '검수_감정': Value('string')}

In [7]:
dataset.num_columns 
# feature 갯수

18

In [8]:
dataset.num_rows    
# row 갯수

10035

In [9]:
##############################
#  Feature 제거
##############################
dataset = dataset.remove_columns(
    ['cid', 'time', 'author', 'channel', 'votes', 'replies', 'heart', 'reply', 'time_parsed', 'source_video_id', 'source_video_title', 'text_length', 'is_not_spam', 'votes_norm', 'replies_norm', 'heart_norm']
)
dataset

Dataset({
    features: ['text', '검수_감정'],
    num_rows: 10035
})

In [10]:
##############################
#  Feature 이름 변경
##############################
dataset = dataset.rename_columns({
    "text":"comment",
    "검수_감정":"labels"
})
dataset

Dataset({
    features: ['comment', 'labels'],
    num_rows: 10035
})

In [11]:
##########################################
# Dataset의 값을 조회(행조회)
# indexing, slicing, fancy indexing
##########################################

# dataset[0]
# dataset[:3]
dataset[[1, 4, 9]]

{'comment': ['팬덤이 계속 보길 원하고, 글로벌적 수요가 분명 타레이블보다 높음에도 경영진의 입맛에 맞지 않다는 이유로 공연을 막고, 광고계약을 막아서 수입창출을 막는 것은 부당한 것 맞음. 법이 제대로 해석해내지못한 부분이 분명 있음\n\n법이 불완전한 부분이 있는데, 부당해도 법이 이러니깐 감내해. 너희가 뭔데 나대는데.. 이런식의 비야냥과 무력한 인간들의 조롱조차도 여태해왔던 용기와 에너지로 이겨내요, PERSON_003.\n일일히 댓글달지 못하지만, 마음으로 응원하고 당신들의 목소리에 귀기울이고 이해하고 있는 사람은 그대들이 생각하는 것보다 훨씬 많아요.\n세상을 바꿀지도 모를 그대들을 응원합니다.  \n 마흔의 나이에도 어린 당신들의 용기를 배웁니다.',
  '해외인터뷰에 혐한발언은 진짜 미친거아님? 왜 가면갈수록 멍청한 발언만 골라가면서 하냐? 멤버들은 어리니까 그렇다쳐도 케어해주는 부모들은 뭔생각인거지??',
  '와 미춌다!! 울 PERSON_004 넘 이뻐!ㅜㅠ💙🩷💛💚💜🐰🐰🐰🐰🐰'],
 'labels': ['긍정', '부정', '긍정']}

In [12]:
############################################################################
#  특정 feature의 값들 조회 - indexing, slicing, fancy indexing
############################################################################
# dataset['comment']
# dataset['comment'][0]
dataset['comment'][:5]
# dataset['comment'][[1, 9, -1]]

# dataset['labels'][:10]

['PERSON_002 예쁘고 귀엽다 언제든지 PERSON_001 응원하고 있을게 화이팅!!!!',
 '팬덤이 계속 보길 원하고, 글로벌적 수요가 분명 타레이블보다 높음에도 경영진의 입맛에 맞지 않다는 이유로 공연을 막고, 광고계약을 막아서 수입창출을 막는 것은 부당한 것 맞음. 법이 제대로 해석해내지못한 부분이 분명 있음\n\n법이 불완전한 부분이 있는데, 부당해도 법이 이러니깐 감내해. 너희가 뭔데 나대는데.. 이런식의 비야냥과 무력한 인간들의 조롱조차도 여태해왔던 용기와 에너지로 이겨내요, PERSON_003.\n일일히 댓글달지 못하지만, 마음으로 응원하고 당신들의 목소리에 귀기울이고 이해하고 있는 사람은 그대들이 생각하는 것보다 훨씬 많아요.\n세상을 바꿀지도 모를 그대들을 응원합니다.  \n 마흔의 나이에도 어린 당신들의 용기를 배웁니다.',
 'PERSON_003 데뷔때부터 팬인데 이젠 나도 쉴드못쳐주겟다.. 하니가 국회 갔을때부터 일이 ㅈㄴ 잘못됐어 ㅆ1ㅂ; 그이후로 본인들이 감당 못할말을 계속 해버리니까 이사단이 나지 진짜 개슬프네..PERSON_003',
 '참고로 PERSON_003 탄핵집회를 선결제해줬다 ㅋㅋ',
 '해외인터뷰에 혐한발언은 진짜 미친거아님? 왜 가면갈수록 멍청한 발언만 골라가면서 하냐? 멤버들은 어리니까 그렇다쳐도 케어해주는 부모들은 뭔생각인거지??']

In [13]:
########################################
#  feature 의 고유값(unique 값) 조회
########################################
dataset.unique("labels")

['긍정', '부정', '건너뛰기', '중립', '불명확', '', '긍정정', '중립립']

In [14]:
############################################
# map()을 이용해 일괄 처리
############################################

label_map = {
    "긍정정":"긍정",
    "중립립":"중립",
    "건너뛰기":"불명확"
}

def update_labels(row):
    """Label `긍정정`을 `긍정`, `중립립`을 `중립`, `건너뛰기`를 `불명확`으로 변경"""
    label = row['labels']
    new_label = label_map.get(label, label)
    return {"labels" : new_label} 

dataset = dataset.map(update_labels)

In [15]:
dataset.unique('labels')

['긍정', '부정', '불명확', '중립', '']

In [16]:
####################################################
# filter()를 이용해 특정 조건의 데이터만 추출 
####################################################
def check_labels(row):
    """Label이 `불명확`, `'' ` 인 행을 제거"""
    flag = row['labels'] != '불명확' and row['labels'] != ''
    return flag

dataset = dataset.filter(check_labels)

In [17]:
dataset

Dataset({
    features: ['comment', 'labels'],
    num_rows: 5571
})

In [18]:
dataset.unique('labels')

['긍정', '부정', '중립']

In [19]:
###############################################
# Dataset 분리
# - shuffle(): 데이터셋을 섞는다.
# - train_test_split(): 데이터셋 분할
###############################################
dataset = dataset.shuffle()

dataset_dict = dataset.train_test_split(
    test_size=0.15,
)


In [20]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['comment', 'labels'],
        num_rows: 4735
    })
    test: Dataset({
        features: ['comment', 'labels'],
        num_rows: 836
    })
})

In [21]:
train_set = dataset_dict['train']
test_set = dataset_dict['test']

In [26]:
train_set
# test_set

Dataset({
    features: ['comment', 'labels'],
    num_rows: 4735
})

In [28]:

####################################################################
# DatasetDict나 Dataset을 huggingface Datahub에 upload 한다.
# - 본인 계정의 Repository에 업로드 하기 때문에 Loging이 필요하다.
#
# Dataset/DatasetDict의 `push_to_hub("Data ID")`
#  - DataID는 "계정/data_id" 로 구성된다.
####################################################################

# 로그인
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv("env")

hf_token = os.getenv('HUGGINGFACE_API_KEY')

login(hf_token)

In [30]:
user_id = "leeyonghyok"
dataset_id = f"{user_id}/youtube_comment_sentiment_dataset_preprocessing"
dataset_dict.push_to_hub(dataset_id)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/leeyonghyok/youtube_comment_sentiment_dataset_preprocessing/commit/a2d848a2f30e92604a3472408433eab432d67d56', commit_message='Upload dataset', commit_description='', oid='a2d848a2f30e92604a3472408433eab432d67d56', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/leeyonghyok/youtube_comment_sentiment_dataset_preprocessing', endpoint='https://huggingface.co', repo_type='dataset', repo_id='leeyonghyok/youtube_comment_sentiment_dataset_preprocessing'), pr_revision=None, pr_num=None)

In [31]:
############################################
#  huggingface에 업로드한 데이터셋 LOAD
############################################

from datasets import load_dataset
load_datasetdict = load_dataset(dataset_id)
load_datasetdict

README.md:   0%|          | 0.00/406 [00:00<?, ?B/s]

c:\Users\Playdata\SKN\SKN31\11_sLLM_파인튜닝\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--leeyonghyok--youtube_comment_sentiment_dataset_preprocessing. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  369kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 62.6kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4735 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/836 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['comment', 'labels'],
        num_rows: 4735
    })
    test: Dataset({
        features: ['comment', 'labels'],
        num_rows: 836
    })
})

In [32]:
##########################################################
# 로컬 컴퓨터에 저장
# 저장: DatasetDict/Dataset의 save_to_disk("저장 경로")
# 불러오기: load_from_disk("저장경로")
###########################################################
dataset_dict.save_to_disk("data/youtube_comment_dataset") 

Saving the dataset (0/1 shards):   0%|          | 0/4735 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/836 [00:00<?, ? examples/s]

In [33]:
# 불러오기
from datasets import load_from_disk
load_dataset_dict2 = load_from_disk("data/youtube_comment_dataset")
load_dataset_dict2

DatasetDict({
    train: Dataset({
        features: ['comment', 'labels'],
        num_rows: 4735
    })
    test: Dataset({
        features: ['comment', 'labels'],
        num_rows: 836
    })
})

In [34]:
#################################################
# Dataset을 다른 형식으로 변환
#
# Dataset.to_xxxxxx()
#################################################
train_set = dataset_dict['train']

##########################
# DataFrame으로 변환
##########################
df_train = train_set.to_pandas()
df_train.head()

,comment,labels
0,근데 진짜 PC 살때는 다른거 다 버리더라도 모니터에 투자하세요. 그래픽카드 30만...,중립
1,PERSON_016 소소한 TMI도 개좋넼ㅋㅋ 더줘 우리는 목마르다,긍정
2,국회의원 월급이나 깎으면 안되나?\n니들은 하는것도 없는데 왜 억단위로 급여를 받는...,부정
3,그냥 개낭비컨텐츠네,부정
4,"PERSON_048 PERSON_051, PERSON_025 해가지고 탑 문파도 하...",중립


In [35]:
###########################################
# 다른 형식을 Dataset으로 변환
#
# Dataset.from_xxxxxx()
###########################################

from datasets import Dataset
dset = Dataset.from_pandas(df_train.head(100))
dset

Dataset({
    features: ['comment', 'labels'],
    num_rows: 100
})

#3.  sLLM 파인튜닝을 위한 데이터 전치리

## 2.1 Dataset Loading

- [Naver 경제, IT 뉴스기사 요약 데이터셋](https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko)
  - NLP 모델 학습을 위해 Naver 뉴스 기사를 크롤링한 데이터셋
  - **2022년 7월 1일 ~ 7월 10일** 기간 동안의 **IT와 경제 기사**를 수집
  - **Train, Test, Validation** Dataset으로 구성됨

In [49]:
from datasets import load_dataset

# Hugging Face 데이터셋 로드
dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")
dataset


DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [50]:
trainset = dataset['train']

In [51]:
df_train = trainset.to_pandas()

In [52]:
# 경제 기사만 조회
df = df_train.query("category=='economy'").reset_index(drop=True)
df.shape

(17088, 7)

In [53]:
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## 2.2 데이터셋 만들기
- 뉴스기사 제목과 뉴스기사를 이용해 그 기사에 영향을 받는 주가종목을 추론하는 모델을 만든다.
- 데이터 구성
  - **입력**: 뉴스 기사 제목, 뉴스 기사
  - **출력**
    1. 뉴스기사가 주식시장에 영향을 주는지 여부
    1. 이 기사에 부정적인 영향을 받는 회사들과 이유
    1. 이 기사에 긍정적인 영향을 받는 회사들과 이유
    1. 뉴스기사 요약
- Dataset 생성
  - Label(정답)을 **LLM을 이용해 생성**한다.
  - **LLM을 이용해 데이터셋을 만든 이후 그 결과를 눈으로 검토해야 한다.**

### 2.2.1 Dataset 생성 Chain 구성

In [63]:
from tqdm import tqdm
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv("env")
# tqdm 반복문 돌릴 때 process를 보여주는 library

True

In [55]:
# 프롬프트 템플릿

template = '''# Instruction
당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.
사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.

## 분석 기준
1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.
2. 영향을 준다면 다음 항목을 출력하세요.
   - `"is_stock_related": true`
   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**
   - 뉴스에 **부정적** 영향을 받는 **회사이름들**
   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**
     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.
     - `None`, 유추, 추정, 일반 논평 금지합니다.
   - **뉴스 내용 요약** (3줄 이내)
3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.
   - `"is_stock_related": false`
   - **뉴스 내용 요약** (3줄 이내)

# 입력 데이터(뉴스기사)

{input}


# 출력 지시사항 (Output Indicator)

- {format_instructions}

## 출력 조건:
- 뉴스에 영향을 받은 회사들은 **반드시 한국 증시에 상장된 종목** 이어야 합니다.
- 뉴스에 있는 내용만 출력결과에 포함시킵니다.
- 긍정/부정 종목은 실제 뉴스기사에 영향을 받는 회사들만 포함하세요.
- 모든 문자열은 큰따옴표(`"`)로 감쌉니다.
- 문자열 안에 따옴표가 필요하면 작은따옴표(`'`)를 사용합니다.
- 모든 키(Key)는 출력 지시사항에 명시된 property들과 정확히 일치해야 합니다.
- `"positive_reasons"` 및 `"negative_reasons"`의 값은 `None`이 될 수 없습니다.
- json format을 잘 지켜 응답데이터를 만듭니다. 배열이나 object의 마지막 항목 뒤에 `,` 를 붙이지 마세요.
- 오직 유효한 JSON 문자열(UTF-8, RFC8259 준수)만 출력합니다.
- 절대 다른 텍스트, 주석, 설명, 코드 블록 표기(```) 들을 추가하면 안 됩니다.

## 출력 예시 (Examples)

### 뉴스가 특정 주식종목들에 **긍정적 영향이 주는 경우**:
{{"is_stock_related": true,
 "negative_reasons": [],
 "negative_stocks": [],
 "positive_reasons": [{{"세라젬": "루게릭병 환우 지원 캠페인 후원과 의료가전 지원 등 사회공헌활동을 통해 기업 이미지와 브랜드 가치가 긍정적으로 부각됨"}}],
 "positive_stocks": ["세라젬"],
 "summary": "세라젬이 루게릭병 환우를 위한 아이스버킷 챌린지 런 행사를 후원하며 의료가전과 건강기능식품 등을 지원했다. 캠페인은 루게릭병 환우 지원과 기부 문화 확산을 목표로 한다. 세라젬은 다양한 사회공헌활동을 지속하고 있다."
}}

### 뉴스의 내용이 특정 주식종목들에 **부정적 영향이 주는 경우**:
{{
    "is_stock_related": true,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [
        "포스코",
        "현대제철"
    ],
    "negative_reasons": [
        {{"포스코": "정부가 수입규제국 조사에 적극 대응하고 비관세장벽 해소를 위해 민관 협력 강화 방침을 밝혀 철강 분야에서 수출 피해 최소화 기대"}},
        {{"현대제철": "철강·금속 품목에 대한 수입규제 대응 강화로 불합리한 무역제한 조치 개선 가능성이 높아져 수출 환경 개선 기대"}}
    ],
    "summary": "산업부는 수입 규제국의 조사에 대응하고 비관세장벽 해소를 위한 협의를 진행했다. 규제 대상 국가는 26개국, 건수는 199건에 달한다."
}}

### **뉴스기사가 주식 종목과 관련 없는 경우**:
{{
    "is_stock_related": false,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [],
    "negative_reasons": [],
    "summary": "정황근 농림축산식품부 장관이 단순가공식품 부가가치세 면제 시행 상황을 점검했다. 된장, 고추장 코너를 방문하며 현장을 살폈다."
}}'''

In [56]:
# Output Parser 정의
class SummarySchema(BaseModel):
    is_stock_related: bool = Field(description="한국 주식 종목과 관련있는 뉴스인지 여부")
    positive_stocks: list[str] = Field(description="뉴스기사에 긍정적인 영향을 받는 회사들의 이름들.")
    positive_reason: list[dict[str,str]] = Field(description='뉴스내용 중 positive_stocks에 있는 각 회사들에 긍정적 영향을 주는 내용. {"회사이름":"긍정적인 이유"}')
    negative_stocks: list[str] = Field(description="뉴스기사에 부정적인 영향을 받는 회사들의 이름들.")
    negative_reason: list[dict[str,str]] = Field(description='뉴스내용 중 negative_stocks에 있는 각 회사들에 부정적 영향을 주는 내용. {"회사이름":"부정적인 이유"}')
    summary: str = Field(description="뉴스 기사 요약")

parser = JsonOutputParser(pydantic_object=SummarySchema)

prompt = ChatPromptTemplate.from_template(
    template=template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Chain 구성
model_name = "gpt-5.4-mini"
model = ChatOpenAI(model=model_name)
chain = prompt | model | parser

#### 2.2.1.1개별 데이터로 Chain 테스트

In [57]:
# 입력 형식: title\n뉴스내용

from pprint import pprint

news_idx = 10
sample_news = df['title'].loc[news_idx]+"\n"+df['document'].loc[news_idx]
print(sample_news)

글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망
경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가 줄어들 것 이라고 설명했습니다. 그러면서 올해 전체 출하량은 14억6천만대 수준에 그칠 것으로 예측했는데요. 종전 전망치인 16억대에서 대폭 낮춰 잡았습니다. 특히 세계 최대 스마트폰 시장인 중국에서 판매량은 18%가 감소할 것으로 전망했는데요. 가트너는 이같은 수요 부진으로 애플을 비롯한 스마트폰 제조사부터 엔비디아 TSMC 같은 반도체 업체까지 압력이 가해질 것이라고 진단했습니다. ◇ EU 가상자산 돈세탁 막는다 유럽연합이 가상자산을 이용한 돈세탁을 막기위해 관련 기업을 규제하는 방안에 잠정 합의했습니다. 잠정안에는 가상자산 업체가 당국에 모든 디지털자산 거래에 대한 신원 확인 정보를 제공하도록 하는 내용이 담겼는데요. 이에 따라 업체들은 관련 개인정보를 확보해야하고 당국이 이를 요구할 경우 제출해야 합니다. 또 거래액이 1천 유로 우리돈 130만 원을 넘길 경우 비인증 거래소가 관리하는 가상자산 지갑도 똑같은 규칙이 적용되는데요. 여기에 더해 송금 규제를 활용해 거래를 상시 추적하고 불법성이 의심되는 거래를 막을 수 있도록 할 방침입니다. 이와 관련해 미국 최대 가상자산 거래소 코인베이스 등 관련 기업 40여 곳은 개인정보 침해 가능성을 언급하며 줄곧 반대 입장을 밝혀왔는데요. 하지만 최근 가상자산 관련 범죄 사례가 급증하고 있는 만큼 규제 움직임이 힘을 얻고 있는 것으로 풀이됩니다. 관련 기관 논의는 오는 30일까지 마무리될 예정인데 이후 EU 위원회와 의회의 승인 절차만 남게 됩니다. ◇ 스피릿 M A 주주투표 또 미뤄 미국 저비용 항

In [58]:
# Chain 실행을 통해 Label 생성
response = chain.invoke({"input":sample_news})

pprint(response)

{'is_stock_related': True,
 'negative_reasons': [{'애플': '가트너가 올해 전세계 스마트폰 판매량이 7% 감소하고 중국 판매량도 18% 줄 것으로 '
                             '전망하면서 스마트폰 제조사에 압력이 가해질 것이라고 언급함'},
                      {'엔비디아': '가트너가 스마트폰 수요 부진의 영향이 스마트폰 제조사뿐 아니라 엔비디아 같은 반도체 '
                               '업체까지 미칠 것이라고 진단함'},
                      {'TSMC': '가트너가 스마트폰 수요 부진의 영향이 스마트폰 제조사뿐 아니라 TSMC 같은 반도체 '
                               '업체까지 미칠 것이라고 진단함'},
                      {'코인베이스': 'EU가 가상자산 돈세탁 방지를 위해 거래 신원 확인 정보 제공과 개인정보 '
                                '확보·제출 의무 등 강한 규제를 추진하면서 관련 기업에 부담이 커짐'},
                      {'텐센트': '월스트리트저널이 텐센트가 지난해에 이어 하반기 대규모 추가 감원을 예고했고 일부 '
                              '사업의 전면 철수까지 검토 중이라고 전함'},
                      {'바이트댄스': '월스트리트저널이 바이트댄스를 비롯한 중국 빅테크들이 지난해 수만명을 줄인 데 이어 '
                                '또 한번 대규모 구조조정을 준비 중이라고 전함'}],
 'negative_stocks': ['애플', '엔비디아', 'TSMC', '코인베이스', '텐센트', '바이트댄스'],
 'positive_reasons': [],
 'positive_stocks': [],
 'summary': '가

### 2.2.2 Label 만들기

1. 뉴스제목(title)과 뉴스기사(document)를 합쳐서 입력데이터를 만든다.
2. 1의  입력데이터를 LLM에 요청해서 답변을 받은 뒤 DataFrame에 추가한다.

In [64]:
# K개 샘플링

sample_nums = 100  
sample_df = df.sample(sample_nums).reset_index(drop=True)
sample_df.shape

(100, 7)

In [66]:
# 뉴스제목(title)과 뉴스기사(document)를 합쳐서 프롬프트를 생성

articles = sample_df['title']+"\n"+sample_df['document']
articles[0]

'서울 아파트 매수심리 8주째 위축\n서울 뉴시스 권창회 기자 서울 아파트 매수심리가 금리 인상 집값 하락 우려 등으로 하락세를 보이고 있다. 한국부동산원에 따르면 6월 넷째 주 27일 기준 서울 아파트 매매수급지수는 지난주 88.1에서 이번주 87.0으로 1포인트 넘게 떨어졌다. 5월 9일부터 8주 연속 하락세다. 사진은 3일 오후 서울 광진구 아파트단지 모습. 2022.07.03. kch0523 newsis.com 사진 영상 제보받습니다 공감언론 뉴시스가 독자 여러분의 소중한 제보를 기다립니다. 뉴스 가치나 화제성이 있다고 판단되는 사진 또는 영상을 뉴시스 사진영상부 n photo newsis.com 로 보내주시면 적극 반영하겠습니다.'

In [67]:
# LLM에 label 생성 요청
label_list = articles.apply(lambda x : chain.invoke({'input':x}))

In [68]:
# 결과 확인
idx = 0
print(articles[idx])
label_list[idx]

서울 아파트 매수심리 8주째 위축
서울 뉴시스 권창회 기자 서울 아파트 매수심리가 금리 인상 집값 하락 우려 등으로 하락세를 보이고 있다. 한국부동산원에 따르면 6월 넷째 주 27일 기준 서울 아파트 매매수급지수는 지난주 88.1에서 이번주 87.0으로 1포인트 넘게 떨어졌다. 5월 9일부터 8주 연속 하락세다. 사진은 3일 오후 서울 광진구 아파트단지 모습. 2022.07.03. kch0523 newsis.com 사진 영상 제보받습니다 공감언론 뉴시스가 독자 여러분의 소중한 제보를 기다립니다. 뉴스 가치나 화제성이 있다고 판단되는 사진 또는 영상을 뉴시스 사진영상부 n photo newsis.com 로 보내주시면 적극 반영하겠습니다.


{'is_stock_related': False,
 'positive_stocks': [],
 'positive_reasons': [],
 'negative_stocks': [],
 'negative_reasons': [],
 'summary': '서울 아파트 매수심리가 금리 인상과 집값 하락 우려로 8주 연속 위축됐다. 서울 아파트 매매수급지수는 88.1에서 87.0으로 하락했다. 기사에는 특정 상장 종목이나 기업이 직접 언급되지 않았다.'}

In [69]:
# Label 타입 변환:  Dictionary 를 str로 변환.
label_list_str = label_list.apply(lambda x : str(x))

In [70]:
# Label을 DataFrame에 "label" 컬럼에 추가.
sample_df['label'] = label_list_str

## 2.3 생성된 데이터셋 저장

1. Local에 pickle 이용해 저장
2. Huggingface data-hub에 upload

### 2.3.1 pickle로 저장

In [71]:

import os
os.makedirs("dataset", exist_ok=True)

output_file = "dataset/sample_df.pkl"

sample_df.to_pickle(
    output_file
)

### 2.3.2 허깅페이스에 업로드
1. login
2. Dataset객체.push_to_hub(dataset_id: str)

In [73]:
from huggingface_hub import login
from datasets import Dataset, load_dataset
from dotenv import load_dotenv
import os

load_dotenv("env")

True

In [74]:
# 로그인
login(os.getenv("HUGGINGFACE_API_KEY"))

In [75]:
# DataFrame을 Dataset으로 변환

dataset = Dataset.from_pandas(sample_df)
dataset

Dataset({
    features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
    num_rows: 100
})

In [76]:
# train/test set으로 분리
dataset_dict = dataset.train_test_split(test_size=0.1)
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 90
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 10
    })
})

In [77]:
# 데이터셋을 Huggingface hub 에 업로드.
dataset_id = "naver_economy_news_stock_instruct_dataset-100_samples"
dataset_dict.push_to_hub(dataset_id)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/leeyonghyok/naver_economy_news_stock_instruct_dataset-100_samples/commit/7fb636dccd493cd9c2362ff8e1df207e14fa10b7', commit_message='Upload dataset', commit_description='', oid='7fb636dccd493cd9c2362ff8e1df207e14fa10b7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/leeyonghyok/naver_economy_news_stock_instruct_dataset-100_samples', endpoint='https://huggingface.co', repo_type='dataset', repo_id='leeyonghyok/naver_economy_news_stock_instruct_dataset-100_samples'), pr_revision=None, pr_num=None)

In [78]:
# Dataset load
user_id = "leeyonghyok" # 본인 Huggingface 사용자명 입력
dataset_id = "naver_economy_news_stock_instruct_dataset-100_samples"
load_dataset_dataset_id = f"{user_id}/{dataset_id}"
load_data = load_dataset(load_dataset_dataset_id)

README.md:   0%|          | 0.00/657 [00:00<?, ?B/s]

c:\Users\Playdata\SKN\SKN31\11_sLLM_파인튜닝\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--leeyonghyok--naver_economy_news_stock_instruct_dataset-100_samples. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  174kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.5kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10 [00:00<?, ? examples/s]

In [79]:
load_data

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 90
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 10
    })
})

In [80]:
# 1500개 sample 로 만든 데이터 셋 로드

from datasets import load_dataset
d_id = "kgmyh/naver_economy_news_stock_instruct_dataset"
dataset = load_dataset(d_id)

README.md:   0%|          | 0.00/622 [00:00<?, ?B/s]

c:\Users\Playdata\SKN\SKN31\11_sLLM_파인튜닝\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--kgmyh--naver_economy_news_stock_instruct_dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.68MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  315kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1350 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

In [81]:
dataset

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 150
    })
})

In [82]:
# 확인
trainset = dataset['train']
testset = dataset['test']

In [83]:
idx = 0
trainset[idx]

{'date': '2022-07-05 13:41:15',
 'category': 'economy',
 'press': '뉴스1 ',
 'title': '대전도시공사 한국서비스 품질 우수기업 인증 획득',
 'document': '대전도시공사 김홍준 경영부사장 오른쪽 이 한국서비스진흥협회 관계자로부터 인증서를 받은 뒤 기념촬영을 하고 있다. 대전도시공사 제공 © 뉴스1 대전 뉴스1 김경훈 기자 대전도시공사는 산업통상자원부에서 주최하고 한국서비스진흥협회에서 주관하는 한국서비스 품질 우수기업 SQ 인증을 획득했다고 5일 밝혔다. SQ는 산업통상자원부 국가기술표준원의 평가지표를 기준으로 Δ고객접점 서비스 운영관리 Δ서비스 경영 성과 등 총 7개 항목 25개 지표에 대한 서류 심사 현장 평가 암행 평가 등 객관적 심사를 거쳐 인증서를 수여하고 널리 공표하는 제도다. 공사는 각종 사업을 추진하면서 적극적으로 시민의 목소리를 반영하고 입주민이 스스로 결정하는 주택 관리 착한 임대인 사업 29년 연속 흑자 경영과 이익배당 사회공헌 프로그램 등에서 높은 점수를 받았다. 공사 관계자는 “한국서비스품질 우수기업 인증 위상에 걸맞게 보다 품질 높은 서비스를 시민에게 제공하겠다”고 말했다.',
 'link': 'https://n.news.naver.com/mnews/article/421/0006198155?sid=101',
 'summary': '5일 5일 대전도시공사는 산업통상자원부에서 주최하고 한국서비스진흥협회에서 주관하는 총 7개 항목 25개 지표에 대한 서류 심사 현장 평가 암행 평가 등 객관적 심사를 거쳐 인증서를 수여하고 널리 공표하는 제도인 한국서비스 품질 우수기업 SQ 인증을 획득했다고 밝혔다.',
 'label': "{'is_stock_related': False, 'positive_stocks': [], 'positive_reasons': [], 'negative_stocks': [], 'negative_reasons': [], 'summary': '대전도시공사가 

In [84]:
testset[idx]

{'date': '2022-07-06 16:28:16',
 'category': 'economy',
 'press': '연합뉴스 ',
 'title': '삼성전자 반도체·스마트폰·TV에 상반기 성과급 100% 지급',
 'document': '최대치인 기본급 100%로 결정…생활가전은 제일 낮은 62.5% 삼성전자 서초사옥 연합뉴스 자료사진 서울 연합뉴스 김철선 기자 삼성전자가 반도체와 스마트폰 TV 등 주력 사업부 소속 임직원들에게 올해 상반기 성과급으로 최대치인 기본급 상여 기초금 의 100%를 지급하기로 했다. 6일 업계에 따르면 삼성전자는 이날 오후 사내망을 통해 직원들에게 상반기 사업부별 목표달성장려금 TAI 지급률을 통보했다. TAI는 성과급 중 하나로 매년 상·하반기 한 차례씩 지급되며 사업부 실적을 토대로 사업 부문과 사업부의 평가를 합쳐 최대 월 기본급의 100%를 지급한다. 1년에 한 번 연봉의 최대 50%까지 받을 수 있는 초과이익성과급 OPI 과 함께 삼성전자의 대표적인 성과급 제도다. 반도체 사업을 담당하는 디바이스솔루션 DS 부문의 메모리반도체 사업부 파운드리 사업부 시스템LSI 사업부는 모두 최대치인 100%를 받는다. 스마트폰 사업부인 MX사업부와 네트워크사업부 TV 사업을 담당하는 영상디스플레이 VD 사업부도 역시 최대치인 100%를 받게 됐다. 반면 냉장고와 세탁기 등 제품을 담당하는 생활가전사업부에는 전사 사업부 중 가장 낮은 수준인 62.5%의 지급률이 통보됐다. 최근 원자재 가격 상승과 물류비 인상 수요 위축 등의 영향으로 예상보다 수익성이 신통치 않았던 영향인 것으로 알려졌다. 삼성전자는 다음날인 7일 올해 2분기 잠정 경영실적을 발표하고 오는 8일 사업부별 지급률에 따라 상반기 TAI를 지급할 예정이다.',
 'link': 'https://n.news.naver.com/mnews/article/001/0013293397?sid=101',
 'summary': '삼성전자가 반도체와 스마트폰 TV 등 주력 사업부 소속 임직